### install dependencies and helpful functions

In [ ]:
!pip install openai-whisper wordfreq transformers accelerate bitsandbytes -q
!apt-get install -y ffmpeg -q

Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [ ]:
import whisper
import subprocess
import re
import json
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

try:
    from wordfreq import zipf_frequency
except ImportError:
    raise ImportError("Run Cell 1 first to install dependencies.")



```
# This is formatted as code
```

### Video Processing

we use dictionary for now, but need to rethink the datastructure later

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define all your videos
videos = {
    'video1': '/content/drive/MyDrive/Colab Notebooks/Copy of EFL teacher.mp4',
    'video2': '/content/drive/MyDrive/Colab Notebooks/Copy of Eugene.mp4',
    'video3': '/content/drive/MyDrive/Colab Notebooks/Copy of yuxin.mp4'
}

# Just change this line to pick which one you want to process
selected = 'video3'

video_path = videos[selected]
print(f"Selected video path: {video_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Selected video path: /content/drive/MyDrive/Colab Notebooks/Copy of yuxin.mp4


In [ ]:
audio_path = "lecture_audio.wav"
result = subprocess.run(
    [
        "ffmpeg", "-i", video_path,
        "-vn",                    # no video
        "-acodec", "pcm_s16le",   # uncompressed PCM
        "-ar", "16000",           # 16 kHz (Whisper's preferred rate)
        "-ac", "1",               # mono
        audio_path, "-y",
        "-loglevel", "quiet",
    ],
    capture_output=True,
)
if result.returncode != 0:
    print("ffmpeg error:", result.stderr.decode())
else:
    print(f"✓ Audio extracted → {audio_path}")

✓ Audio extracted → lecture_audio.wav


In [ ]:
#Speech to Text

In [ ]:
MODEL_SIZE = "base"
LANGUAGE   = "en"

print(f"Loading Whisper '{MODEL_SIZE}' model…")
model = whisper.load_model(MODEL_SIZE)

print("Transcribing… (may take a few minutes depending on video length)")
result = model.transcribe(audio_path, language=LANGUAGE, verbose=False)

transcript = result["text"].strip()
segments   = result["segments"]          # each segment has start/end timestamps
duration_s = segments[-1]["end"] if segments else 1

print(f"\n✓ Transcription complete")
print(f"  Duration  : {duration_s:.1f}s  ({duration_s/60:.1f} min)")
print(f"  Words     : {len(transcript.split())}")
print("\n--- TRANSCRIPT PREVIEW (first 400 chars) ---")
print(transcript[:400] + ("…" if len(transcript) > 400 else ""))

Loading Whisper 'base' model…
Transcribing… (may take a few minutes depending on video length)


100%|██████████| 7458/7458 [00:03<00:00, 2254.36frames/s]


✓ Transcription complete
  Duration  : 67.0s  (1.1 min)
  Words     : 171

--- TRANSCRIPT PREVIEW (first 400 chars) ---
Hi, class. Welcome to our Community Impact Program again. And today is our week six. We're going to talk about lesson two. While we're still waiting for more people to join us, why don't we start with our agenda? We'll first review what we just learned from yesterday. Since I was not here, I'd like to have someone in the class to help me review what we learned. And the second thing we're going to …


# **Added Validation with NLI classifier (deberta-v3-small) if the transcript is not teacher's instruction the system will not continue**

# **Didn't work, replaced with Qwen model. there is no domain specific model for teaching instruction**

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from IPython.display import display, HTML as IHTML

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = "nf4",
    bnb_4bit_compute_dtype    = torch.float16,
    bnb_4bit_use_double_quant = True,
)

print(f"Loading {MODEL_ID} in 4-bit… (downloads ~4 GB on first run)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = "auto",
)
model.eval()
print("✓ Model ready —", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

# ── Validate transcript ──────────────────────────────────────────

val_messages = [
    {"role": "system", "content": "You are a classifier. Respond with only YES or NO, nothing else."},
    {"role": "user", "content": f"""Read the following text and answer YES or NO only.
Is the speaker THEMSELVES a teacher who is directly talking TO students right now?
- YES = the speaker is a teacher giving a live lesson (e.g. "Okay class, let's begin", "Can you tell me...")
- NO = the speaker is talking ABOUT a lesson or teacher (e.g. "the teacher did...", "in terms of scaffolding...", "I think she...")
Text:
{' '.join(transcript.split()[:512])}
Answer YES or NO only:"""},
]

val_text   = tokenizer.apply_chat_template(val_messages, tokenize=False, add_generation_prompt=True)
val_inputs = tokenizer([val_text], return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **val_inputs,
        max_new_tokens          = 5,
        do_sample               = False,
        temperature             = None,
        top_p                   = None,
        pad_token_id            = tokenizer.eos_token_id,
        output_scores           = True,       # ← 新增
        return_dict_in_generate = True,       # ← 新增
    )

val_ids    = output.sequences
val_answer = tokenizer.decode(val_ids[0][val_inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().upper()

# ── Confidence score ─────────────────────────────────────────────
first_token_logits = output.scores[0][0]
probs = torch.softmax(first_token_logits, dim=-1)

yes_token_id = tokenizer.encode("YES", add_special_tokens=False)[0]
no_token_id  = tokenizer.encode("NO",  add_special_tokens=False)[0]

yes_prob   = probs[yes_token_id].item()
no_prob    = probs[no_token_id].item()
confidence = yes_prob / (yes_prob + no_prob)  # normalized to YES vs NO only

print(f"\n  Transcript validation : {val_answer}")
print(f"  Confidence (YES)      : {confidence:.2%}")

if "YES" not in val_answer:
    display(IHTML("""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
                max-width:600px;margin:24px auto;padding:24px 28px;
                background:#faece7;border-radius:12px;border:1px solid #f0997b">
      <div style="font-size:18px;font-weight:600;color:#993C1D;margin-bottom:8px">
        &#9888;&#65039; Invalid input — pipeline stopped
      </div>
      <div style="font-size:14px;color:#712B13;line-height:1.6;margin-bottom:16px">
        This transcript does not appear to be a <strong>live classroom teaching session</strong>.
      </div>
      <div style="font-size:13px;color:#993C1D;background:#f5c4b3;
                  border-radius:8px;padding:12px 14px">
        Please provide a direct classroom teaching transcript and re-run from Cell 4.
      </div>
    </div>"""))
    raise SystemExit("Stopping: input is not a valid teaching transcript.")

print("  ✓ Confirmed teaching transcript — continuing pipeline")

Loading Qwen/Qwen2.5-7B-Instruct in 4-bit… (downloads ~4 GB on first run)


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✓ Model ready — Tesla T4

  Transcript validation : YES
  Confidence (YES)      : 100.00%
  ✓ Confirmed teaching transcript — continuing pipeline


Text Analysis (filler words, instructino phrases and word level)

In [ ]:
# Add or remove words/phrases from this list to suit your use case.
FILLER_WORDS = [
    # Hesitation sounds
    "um", "uh", "er", "ah",
    # Over-used discourse markers
    "like", "you know", "i mean", "i guess",
    "basically", "literally", "actually",
    "right", "okay so", "so yeah",
    "kind of", "sort of",
    "well", "anyway", "you see",
]

def analyze_filler_words(text: str) -> dict:
    """Count filler words and return rate per 100 words."""
    text_lower = text.lower()
    counts = {}
    total  = 0

    # Sort by length (descending) so multi-word fillers are matched first
    for filler in sorted(FILLER_WORDS, key=len, reverse=True):
        pattern = r"(?<!\w)" + re.escape(filler) + r"(?!\w)"
        n = len(re.findall(pattern, text_lower))
        if n:
            counts[filler] = n
            total += n

    word_count = len(text.split())
    rate = round(total / word_count * 100, 2) if word_count else 0

    return {
        "total_filler_count"  : total,
        "total_words"         : word_count,
        "filler_rate_pct"     : rate,          # fillers per 100 words
        "breakdown"           : dict(sorted(counts.items(), key=lambda x: -x[1])),
    }

In [ ]:
# Grouped by pedagogical function for easy interpretation.
INSTRUCTION_PATTERNS = [
    # Sequencing / transitions
    (r"\bnow let'?s\b",            "Now let's…"),
    (r"\blet'?s\b",                "Let's…"),
    (r"\bfirst[,\s]",              "First…"),
    (r"\bsecond(ly)?[,\s]",        "Second(ly)…"),
    (r"\bthird(ly)?[,\s]",         "Third(ly)…"),
    (r"\bnext[,\s]",               "Next…"),
    (r"\bfinally[,\s]",            "Finally…"),
    (r"\bto begin\b",              "To begin"),

    # Direct student directives
    (r"\bplease\b",                "Please…"),
    (r"\bmake sure\b",             "Make sure…"),
    (r"\bremember (to|that)\b",    "Remember to/that…"),
    (r"\bi want you to\b",         "I want you to…"),
    (r"\byou should\b",            "You should…"),
    (r"\btry to\b",                "Try to…"),
    (r"\bgo ahead\b",              "Go ahead…"),
    (r"\bpay attention\b",         "Pay attention…"),

    # Explanation / clarification signals
    (r"\bfor example\b",           "For example"),
    (r"\bfor instance\b",          "For instance"),
    (r"\bin other words\b",        "In other words"),
    (r"\bwhat this means\b",       "What this means"),
    (r"\bnotice that\b",           "Notice that"),
    (r"\blook at\b",               "Look at"),
    (r"\bto summarize\b",          "To summarize"),
    (r"\bto recap\b",              "To recap"),
    (r"\bthe key (point|idea|takeaway)\b", "The key point/idea"),
]

def analyze_instruction_phrases(text: str, duration_seconds: float) -> dict:
    """Count instruction phrases and return rate per minute."""
    text_lower = text.lower()
    counts = {}
    total  = 0

    for pattern, label in INSTRUCTION_PATTERNS:
        n = len(re.findall(pattern, text_lower))
        if n:
            counts[label] = n
            total += n

    duration_min  = max(duration_seconds / 60, 0.01)
    rate_per_min  = round(total / duration_min, 2)

    return {
        "total_instruction_phrases" : total,
        "rate_per_minute"           : rate_per_min,
        "breakdown"                 : dict(sorted(counts.items(), key=lambda x: -x[1])),
    }


Run analysis and report

In [ ]:

# ── 5A: Filler word analysis ─────────────────────────────────────
# Add or remove words/phrases from this list to suit your use case.
FILLER_WORDS = [
    # Hesitation sounds
    "um", "uh", "er", "ah",
    # Over-used discourse markers
    "like", "you know", "i mean", "i guess",
    "basically", "literally", "actually",
    "right", "okay so", "so yeah",
    "kind of", "sort of",
    "well", "anyway", "you see",
]

def analyze_filler_words(text: str) -> dict:
    """Count filler words and return rate per 100 words."""
    text_lower = text.lower()
    counts = {}
    total  = 0

    # Sort by length (descending) so multi-word fillers are matched first
    for filler in sorted(FILLER_WORDS, key=len, reverse=True):
        pattern = r"(?<!\w)" + re.escape(filler) + r"(?!\w)"
        n = len(re.findall(pattern, text_lower))
        if n:
            counts[filler] = n
            total += n

    word_count = len(text.split())
    rate = round(total / word_count * 100, 2) if word_count else 0

    return {
        "total_filler_count"  : total,
        "total_words"         : word_count,
        "filler_rate_pct"     : rate,          # fillers per 100 words
        "breakdown"           : dict(sorted(counts.items(), key=lambda x: -x[1])),
    }


# ── 5B: Clear instruction phrase analysis ────────────────────────
# Each tuple is (regex_pattern, display_label).
# Grouped by pedagogical function for easy interpretation.
INSTRUCTION_PATTERNS = [
    # Sequencing / transitions
    (r"\bnow let'?s\b",            "Now let's…"),
    (r"\blet'?s\b",                "Let's…"),
    (r"\bfirst[,\s]",              "First…"),
    (r"\bsecond(ly)?[,\s]",        "Second(ly)…"),
    (r"\bthird(ly)?[,\s]",         "Third(ly)…"),
    (r"\bnext[,\s]",               "Next…"),
    (r"\bfinally[,\s]",            "Finally…"),
    (r"\bto begin\b",              "To begin"),

    # Direct student directives
    (r"\bplease\b",                "Please…"),
    (r"\bmake sure\b",             "Make sure…"),
    (r"\bremember (to|that)\b",    "Remember to/that…"),
    (r"\bi want you to\b",         "I want you to…"),
    (r"\byou should\b",            "You should…"),
    (r"\btry to\b",                "Try to…"),
    (r"\bgo ahead\b",              "Go ahead…"),
    (r"\bpay attention\b",         "Pay attention…"),

    # Explanation / clarification signals
    (r"\bfor example\b",           "For example"),
    (r"\bfor instance\b",          "For instance"),
    (r"\bin other words\b",        "In other words"),
    (r"\bwhat this means\b",       "What this means"),
    (r"\bnotice that\b",           "Notice that"),
    (r"\blook at\b",               "Look at"),
    (r"\bto summarize\b",          "To summarize"),
    (r"\bto recap\b",              "To recap"),
    (r"\bthe key (point|idea|takeaway)\b", "The key point/idea"),
]

def analyze_instruction_phrases(text: str, duration_seconds: float) -> dict:
    """Count instruction phrases and return rate per minute."""
    text_lower = text.lower()
    counts = {}
    total  = 0

    for pattern, label in INSTRUCTION_PATTERNS:
        n = len(re.findall(pattern, text_lower))
        if n:
            counts[label] = n
            total += n

    duration_min  = max(duration_seconds / 60, 0.01)
    rate_per_min  = round(total / duration_min, 2)

    return {
        "total_instruction_phrases" : total,
        "rate_per_minute"           : rate_per_min,
        "breakdown"                 : dict(sorted(counts.items(), key=lambda x: -x[1])),
    }


# ── 5C: Vocabulary level analysis (using wordfreq Zipf scores) ────
#
# Zipf frequency = log10(occurrences per billion words in English).
# It's a reliable, unsupervised proxy for CEFR word difficulty:
#
#   Zipf ≥ 6.0  →  Very high frequency  →  ~A1/A2  (e.g. go, big, ask)
#   5.0–6.0     →  High frequency       →  ~B1/B2  (e.g. explain, result)
#   4.0–5.0     →  Medium frequency     →  ~C1     (e.g. classify, derive)
#   < 4.0       →  Low frequency        →  ~C2+    (e.g. orthogonal, heuristic)
#
# Note: this is a rough proxy, not an official CEFR mapping.

LEVEL_BANDS = {
    "A1/A2  (basic)"         : (6.0, 10.0),
    "B1/B2  (intermediate)"  : (5.0,  6.0),
    "C1     (advanced)"      : (4.0,  5.0),
    "C2+    (very advanced)" : (0.0,  4.0),
}

STOP_WORDS = {
    "the","a","an","is","was","are","were","be","been","being",
    "have","has","had","do","does","did","will","would","could",
    "should","may","might","shall","and","or","but","in","on",
    "at","to","for","of","with","by","from","up","about","into",
    "through","during","before","after","above","below","between",
    "i","you","he","she","it","we","they","me","him","her","us","them",
    "my","your","his","its","our","their","this","that","these","those",
    "what","which","who","when","where","why","how","all","each",
    "just","so","also","then","now","very","too","not","no","if",
    "as","than","more","most","some","any","there","here","can",
    "get","got","go","going","make","made","say","said","know",
    "think","see","come","take","use","want","look","need","tell",
}

def analyze_vocabulary_level(text: str) -> dict:
    """Classify content words by Zipf frequency band."""
    words         = re.findall(r"\b[a-z]+\b", text.lower())
    content_words = [w for w in words if w not in STOP_WORDS and len(w) > 2]

    # Score each unique word once
    word_scores = {w: zipf_frequency(w, "en") for w in set(content_words)}

    level_token_counts = {lvl: 0 for lvl in LEVEL_BANDS}
    level_word_sets    = {lvl: [] for lvl in LEVEL_BANDS}

    for word, score in word_scores.items():
        freq_in_transcript = content_words.count(word)
        for lvl, (lo, hi) in LEVEL_BANDS.items():
            if lo <= score < hi:
                level_token_counts[lvl] += freq_in_transcript
                level_word_sets[lvl].append(word)
                break

    total_tokens = sum(level_token_counts.values()) or 1
    level_pct = {
        lvl: round(c / total_tokens * 100, 1)
        for lvl, c in level_token_counts.items()
    }

    # Top advanced words (C1+) by frequency in the transcript
    advanced_words = {
        w: content_words.count(w)
        for w, s in word_scores.items()
        if s < 5.0
    }
    top_advanced = sorted(advanced_words.items(), key=lambda x: -x[1])[:20]

    return {
        "total_content_words"      : sum(level_token_counts.values()),
        "level_distribution_pct"   : level_pct,
        "level_word_examples"      : {lvl: ws[:8] for lvl, ws in level_word_sets.items()},
        "top_advanced_words"       : top_advanced,   # list of (word, count)
    }


In [ ]:
filler_r      = analyze_filler_words(transcript)
instruction_r = analyze_instruction_phrases(transcript, duration_s)
vocab_r       = analyze_vocabulary_level(transcript)


def bar(n: int, scale: int = 1) -> str:
    return "█" * min(int(n / scale), 40)

SEP = "=" * 58

print(f"\n{SEP}")
print("  VIDEO TEACHING ANALYSIS REPORT")
print(f"  File: {video_path}")
print(f"  Duration: {duration_s/60:.1f} min  |  Words: {filler_r['total_words']}")
print(SEP)

# ── Section 1 ────────────────────────────────────────────────────
print(f"\n  1. FILLER WORDS")
print(f"     Total fillers : {filler_r['total_filler_count']}")
print(f"     Filler rate   : {filler_r['filler_rate_pct']}%  (per 100 words)")
print(f"     Benchmark     : <2% excellent | 2-5% good | >5% high")
print()
for word, count in filler_r["breakdown"].items():
    print(f"     {word:<16} {count:>3}  {bar(count)}")

# ── Section 2 ────────────────────────────────────────────────────
print(f"\n  2. CLEAR INSTRUCTION PHRASES")
print(f"     Total uses      : {instruction_r['total_instruction_phrases']}")
print(f"     Rate per minute : {instruction_r['rate_per_minute']}")
print(f"     Benchmark       : ~2–4/min is a well-scaffolded lesson")
print()
for phrase, count in instruction_r["breakdown"].items():
    print(f"     {phrase:<30} {count:>3}  {bar(count)}")

# ── Section 3 ────────────────────────────────────────────────────
print(f"\n  3. VOCABULARY LEVEL (content words only)")
print(f"     Total content words : {vocab_r['total_content_words']}")
print()
for lvl, pct in vocab_r["level_distribution_pct"].items():
    print(f"     {lvl:<30} {pct:>5.1f}%  {bar(pct, scale=2)}")

print(f"\n     Top advanced words used (C1+):")
for word, count in vocab_r["top_advanced_words"][:15]:
    print(f"       {word:<22} ×{count}")

print(f"\n{SEP}\n")




  VIDEO TEACHING ANALYSIS REPORT
  File: /content/drive/MyDrive/Colab Notebooks/Copy of yuxin.mp4
  Duration: 1.1 min  |  Words: 171

  1. FILLER WORDS
     Total fillers : 2
     Filler rate   : 1.17%  (per 100 words)
     Benchmark     : <2% excellent | 2-5% good | >5% high

     right              1  █
     like               1  █

  2. CLEAR INSTRUCTION PHRASES
     Total uses      : 4
     Rate per minute : 3.58
     Benchmark       : ~2–4/min is a well-scaffolded lesson

     Let's…                           1  █
     First…                           1  █
     Second(ly)…                      1  █
     Remember to/that…                1  █

  3. VOCABULARY LEVEL (content words only)
     Total content words : 74

     A1/A2  (basic)                  12.2%  ██████
     B1/B2  (intermediate)           59.5%  █████████████████████████████
     C1     (advanced)               24.3%  ████████████
     C2+    (very advanced)           4.1%  ██

     Top advanced words used (C1+):
    

### HTML REPORT DOCUMENTED

In [ ]:
# ── Suitable level helper ─────────────────────────────────────
def get_suitable_level(level_dist):
    level_weights = {
        "A1/A2  (basic)"        : 1,
        "B1/B2  (intermediate)" : 2,
        "C1     (advanced)"     : 3,
        "C2+    (very advanced)": 4,
    }
    level_labels = {1: "A1/A2", 2: "B1/B2", 3: "C1", 4: "C2+"}

    total   = sum(level_dist.values())
    max_pct = max(level_dist.values(), default=0)
    c1_plus = level_dist.get("C1     (advanced)", 0) + level_dist.get("C2+    (very advanced)", 0)

    if total == 0:
        return "N/A", ""

    weighted = sum(level_weights[k] * v for k, v in level_dist.items()) / total

    if max_pct < 40:
        label = "Mixed level"
        flag  = "Wide vocabulary range across levels"
    else:
        label = f"Suitable for {level_labels[max(1, min(4, round(weighted)))]}"
        flag  = ""

    if c1_plus >= 40:
        flag = "⚠️ May challenge B1/B2 students"

    return label, flag


# ── build_html_report ─────────────────────────────────────────
def build_html_report(
    video_path, duration_s,
    filler_r, instruction_r, vocab_r,
    transcript,
) -> str:

    duration_min = duration_s / 60
    word_count   = filler_r["total_words"]
    filler_count = filler_r["total_filler_count"]
    filler_pct   = filler_r["filler_rate_pct"]
    instr_total  = instruction_r["total_instruction_phrases"]
    instr_rpm    = instruction_r["rate_per_minute"]

    # ── Filler words bar chart rows ───────────────────────────
    max_filler = max(filler_r["breakdown"].values(), default=1)
    filler_rows = ""
    for word, count in list(filler_r["breakdown"].items())[:12]:
        pct = count / max_filler * 100
        filler_rows += f"""
        <div class="bar-row">
          <span class="bar-label">{word}</span>
          <div class="bar-track">
            <div class="bar-fill fill-coral" style="width:{pct:.1f}%"></div>
          </div>
          <span class="bar-count">{count}</span>
        </div>"""

    # ── Instruction phrases bar chart rows ────────────────────
    max_instr = max(instruction_r["breakdown"].values(), default=1)
    instr_rows = ""
    for phrase, count in list(instruction_r["breakdown"].items())[:12]:
        pct = count / max_instr * 100
        instr_rows += f"""
        <div class="bar-row">
          <span class="bar-label">{phrase}</span>
          <div class="bar-track">
            <div class="bar-fill fill-teal" style="width:{pct:.1f}%"></div>
          </div>
          <span class="bar-count">{count}</span>
        </div>"""

    # ── Vocabulary stacked bar ────────────────────────────────
    vocab_colors = {
        "A1/A2  (basic)"         : ("#00BCD4", "#e0f7fa"),
        "B1/B2  (intermediate)"  : ("#4CAF50", "#e8f5e9"),
        "C1     (advanced)"      : ("#FDD835", "#fffde7"),
        "C2+    (very advanced)" : ("#FBC02D", "#fff8e1"),
    }
    stacked_segments = ""
    legend_items     = ""
    for lvl, pct in vocab_r["level_distribution_pct"].items():
        color, bg = vocab_colors[lvl]
        stacked_segments += (
            f'<div class="stack-seg" style="width:{pct}%;background:{color}" '
            f'title="{lvl}: {pct}%"></div>'
        )
        legend_items += (
            f'<span class="legend-dot" style="background:{color}"></span>'
            f'<span class="legend-label">{lvl.strip()}</span>'
            f'<span class="legend-pct">{pct}%</span>'
        )

    # ── Advanced word chips ───────────────────────────────────
    adv_chips = ""
    for word, count in vocab_r["top_advanced_words"][:18]:
        adv_chips += f'<span class="chip">{word} <em>×{count}</em></span>'

    # ── Filler-rate colour signal ─────────────────────────────
    if filler_pct < 2:
        filler_signal = ("✓ Excellent", "#0F6E56")
    elif filler_pct < 5:
        filler_signal = ("◎ Good", "#854F0B")
    else:
        filler_signal = ("△ High", "#993C1D")

    # ── Suitable level badge ──────────────────────────────────
    suitable_label, suitable_flag = get_suitable_level(vocab_r["level_distribution_pct"])
    suitable_flag_html = (
        f'<span style="margin-left:12px;font-size:12px;color:#854F0B">{suitable_flag}</span>'
        if suitable_flag else ""
    )

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<title>Teaching Analysis Report</title>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
    background: #f5f4ef;
    color: #2c2c2a;
    padding: 32px 24px;
  }}
  h1 {{ font-size: 20px; font-weight: 600; margin-bottom: 4px; }}
  .meta {{ font-size: 13px; color: #6b6a63; margin-bottom: 28px; }}

  /* Summary cards */
  .cards {{ display: flex; gap: 14px; flex-wrap: wrap; margin-bottom: 28px; }}
  .card {{
    flex: 1; min-width: 140px;
    background: #fff;
    border-radius: 12px;
    padding: 18px 20px;
    border: 1px solid #e2e0d8;
  }}
  .card-val  {{ font-size: 28px; font-weight: 700; line-height: 1; }}
  .card-sub  {{ font-size: 12px; color: #6b6a63; margin-top: 4px; }}
  .card-sig  {{ font-size: 12px; font-weight: 600; margin-top: 6px; }}

  /* Sections */
  .section {{
    background: #fff;
    border-radius: 12px;
    padding: 22px 24px;
    margin-bottom: 18px;
    border: 1px solid #e2e0d8;
  }}
  .section h2 {{
    font-size: 14px; font-weight: 600;
    margin-bottom: 16px;
    padding-bottom: 10px;
    border-bottom: 1px solid #eeecea;
  }}

  /* Suitable level badge */
  .level-badge {{
    display: inline-block;
    background: #2c2c2a;
    color: #fff;
    font-size: 12px;
    font-weight: 600;
    padding: 3px 10px;
    border-radius: 20px;
    margin-right: 8px;
    vertical-align: middle;
  }}

  /* Bar charts */
  .bar-row {{
    display: flex; align-items: center;
    gap: 10px; margin-bottom: 8px; font-size: 13px;
  }}
  .bar-label {{
    width: 180px; flex-shrink: 0;
    color: #3d3d3a; white-space: nowrap;
    overflow: hidden; text-overflow: ellipsis;
  }}
  .bar-track {{
    flex: 1; height: 10px;
    background: #f0eeea; border-radius: 5px; overflow: hidden;
  }}
  .bar-fill  {{ height: 100%; border-radius: 5px; transition: width .3s; }}
  .fill-coral {{ background: #D85A30; }}
  .fill-teal  {{ background: #1D9E75; }}
  .bar-count {{ width: 28px; text-align: right; color: #888780; }}

  /* Vocabulary stacked bar */
  .stack-bar {{
    display: flex; height: 20px;
    border-radius: 6px; overflow: hidden; margin-bottom: 14px;
  }}
  .stack-seg {{ height: 100%; transition: width .3s; }}
  .legend {{ display: flex; flex-wrap: wrap; gap: 6px 18px; font-size: 13px; }}
  .legend-dot {{
    display: inline-block;
    width: 10px; height: 10px; border-radius: 3px;
    vertical-align: middle; margin-right: 4px;
  }}
  .legend-label {{ color: #3d3d3a; }}
  .legend-pct {{ color: #888780; margin-left: 4px; }}

  /* Word chips */
  .chips {{ display: flex; flex-wrap: wrap; gap: 6px; margin-top: 14px; }}
  .chip {{
    background: #f5f3ee; border: 1px solid #e2e0d8;
    border-radius: 20px; padding: 3px 12px;
    font-size: 12px; color: #3d3d3a;
  }}
  .chip em {{ color: #888780; font-style: normal; }}

  /* Transcript */
  .transcript-box {{
    background: #f9f8f5;
    border: 1px solid #e2e0d8;
    border-radius: 8px;
    padding: 14px 16px;
    font-size: 13px;
    line-height: 1.7;
    color: #3d3d3a;
    max-height: 320px;
    overflow-y: auto;
    white-space: pre-wrap;
    word-break: break-word;
  }}

  /* Scaffolding (injected) */
  .summary-text {{ font-size: 14px; line-height: 1.7; color: #3d3d3a; margin-bottom: 18px; }}
  .finding {{ padding: 10px 14px; border-radius: 8px; margin-bottom: 8px; font-size: 13px; }}
  .strength {{ background: #f0faf5; border-left: 3px solid #1D9E75; }}
  .growth   {{ background: #fdf6ee; border-left: 3px solid #BA7517; }}
  .finding-point    {{ font-weight: 500; margin-bottom: 4px; }}
  .finding-evidence {{ font-size: 12px; }}
  .standout {{
    background: #eef4fb; border-radius: 8px; padding: 12px 16px;
    font-size: 13px; color: #185FA5; margin-top: 16px;
    border-left: 3px solid #185FA5;
  }}
  .two-col {{ display: flex; gap: 16px; }}
  .two-col > div {{ flex: 1; }}
</style>
</head>
<body>

<h1>Teaching Analysis Report</h1>
<p class="meta">
  {video_path} &nbsp;·&nbsp;
  {duration_min:.1f} min &nbsp;·&nbsp;
  {word_count} words
</p>

<!-- Summary cards -->
<div class="cards">
  <div class="card">
    <div class="card-val">{filler_count}</div>
    <div class="card-sub">filler words</div>
    <div class="card-sig" style="color:{filler_signal[1]}">{filler_signal[0]} — {filler_pct}% rate</div>
  </div>
  <div class="card">
    <div class="card-val">{instr_total}</div>
    <div class="card-sub">instruction phrases</div>
    <div class="card-sig" style="color:#185FA5">{instr_rpm}/min</div>
  </div>
  <div class="card">
    <div class="card-val">{vocab_r["level_distribution_pct"].get("A1/A2  (basic)", 0):.0f}%</div>
    <div class="card-sub">basic vocab (A1/A2)</div>
    <div class="card-sig" style="color:#6b6a63">content words</div>
  </div>
  <div class="card">
    <div class="card-val">{vocab_r["level_distribution_pct"].get("C1     (advanced)", 0) + vocab_r["level_distribution_pct"].get("C2+    (very advanced)", 0):.0f}%</div>
    <div class="card-sub">advanced vocab (C1+)</div>
    <div class="card-sig" style="color:#6b6a63">content words</div>
  </div>
</div>

<!-- ① Transcript -->
<div class="section">
  <h2>① Transcript</h2>
  <div class="transcript-box">{transcript}</div>
</div>

<!-- ② Vocabulary level -->
<div class="section">
  <h2>② Vocabulary level distribution &nbsp;<span style="font-weight:400;color:#6b6a63">— content words only</span></h2>
  <div style="margin-bottom:14px">
    <span class="level-badge">{suitable_label}</span>
    {suitable_flag_html}
  </div>
  <div class="stack-bar">{stacked_segments}</div>
  <div class="legend">{legend_items}</div>
  <div class="chips">{adv_chips}</div>
  <p style="font-size:11px;color:#aaa;margin-top:12px">
    Levels estimated via wordfreq Zipf frequency score — approximate CEFR mapping, not official.
  </p>
</div>

<!-- ③ Scaffolding — injected here -->
<!-- SCAFFOLDING_PLACEHOLDER -->

<!-- ④ Filler words -->
<div class="section">
  <h2>④ Filler words &nbsp;<span style="font-weight:400;color:#6b6a63">— top occurrences</span></h2>
  {filler_rows if filler_rows else '<p style="color:#888;font-size:13px">No filler words detected.</p>'}
</div>

<!-- ⑤ Instruction phrases -->
<div class="section">
  <h2>⑤ Instruction phrases &nbsp;<span style="font-weight:400;color:#6b6a63">— top occurrences</span></h2>
  {instr_rows if instr_rows else '<p style="color:#888;font-size:13px">No instruction phrases detected.</p>'}
</div>

</body>
</html>"""
    return html


# ── build_html_report is defined above and called in Cell 15 ────
print("✓ Text analysis complete — run Cell 12 onwards for scaffolding analysis")

✓ Text analysis complete — run Cell 12 onwards for scaffolding analysis


## Scaffolding analysis powered by Qwen

In [ ]:
SYSTEM_PROMPT = """You are an expert instructional ESL/EFL coach specialising in adult learners language teaching.
Evaluate the SCAFFOLDING quality of the lesson transcript provided by the user.

IMPORTANT: Before giving any feedback, first verify the input is a
classroom teaching transcript — i.e. it contains a teacher directly
instructing students.

If the input is NOT a teaching transcript (e.g. it is someone's
reflection, analysis, or commentary about a lesson), respond only with:
{"error": "Input does not appear to be a teaching transcript. Please provide the actual classroom transcript."}

Do not attempt to give feedback on non-transcript input.

Scaffolding dimensions to consider:
1. Sequencing — Are concepts introduced in a logical, building-block order?
2. Prior knowledge activation — Does the teacher connect new material to what students already know?
3. Worked examples & modelling — Does the teacher demonstrate before asking students to do?
4. Gradual release — Is there a shift from teacher-led to student-led activity?
5. Checking for understanding — Does the teacher pause to verify comprehension?
6. Language support — Are new terms defined and explained in accessible language?

You MUST respond with ONLY a valid JSON object. No markdown, no explanation, no text before or after.
Use exactly this structure:
{
  "summary": "<2-3 sentence overall narrative>",
  "strengths": [
    {"point": "<specific strength>", "evidence": "<quote or paraphrase from transcript>"}
  ],
  "areas_for_growth": [
    {"point": "<specific gap>", "evidence": "<quote or paraphrase, or not observed>"}
  ],
  "standout_moment": "<description of the single best scaffolding moment>"
}
Aim for 2-4 strengths and 2-3 areas for growth."""

# Trim transcript to ~3000 words — enough for scaffolding analysis,
# keeps generation fast (~30-45s on T4).
# Increase to 6000 if you want the model to see more of the lesson.
MAX_WORDS     = 3000
transcript_in = " ".join(transcript.split()[:MAX_WORDS])
if len(transcript.split()) > MAX_WORDS:
    print(f"Note: transcript trimmed to {MAX_WORDS} words for speed.")

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": f"TRANSCRIPT:\n{transcript_in}"},
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize          = False,
    add_generation_prompt = True,
)
inputs = tokenizer([text], return_tensors="pt").to(model.device)

print("Running Qwen scaffolding analysis… (~30-60s on T4)")
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens  = 800,
        do_sample       = False,          # greedy — more reliable JSON
        temperature     = None,
        top_p           = None,
        repetition_penalty = 1.1,
        pad_token_id    = tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens
new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
raw     = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

# ── Robust JSON extraction ────────────────────────────────────────
# Qwen sometimes wraps output in ```json … ``` — strip it
if "```" in raw:
    raw = re.sub(r"```[a-z]*\n?", "", raw).replace("```", "").strip()

# Find the outermost {...} block in case model adds preamble text
match = re.search(r"\{.*\}", raw, re.DOTALL)
if not match:
    raise ValueError(f"No JSON found in model output:\n{raw[:500]}")

try:
    scaffolding = json.loads(match.group())
except json.JSONDecodeError as e:
    raise ValueError(f"JSON parse error: {e}\nRaw output:\n{raw[:500]}")

print(f"✓ Done — scaffolding analysis complete")

Running Qwen scaffolding analysis… (~30-60s on T4)


KeyboardInterrupt: 

# Qwen sometimes wraps output in ```json … ``` — strip it


In [ ]:
if "```" in raw:
    raw = re.sub(r"```[a-z]*\n?", "", raw).replace("```", "").strip()

# Find the outermost {...} block in case model adds preamble text
match = re.search(r"\{.*\}", raw, re.DOTALL)
if not match:
    raise ValueError(f"No JSON found in model output:\n{raw[:500]}")

try:
    scaffolding = json.loads(match.group())
except json.JSONDecodeError as e:
    raise ValueError(f"JSON parse error: {e}\nRaw output:\n{raw[:500]}")

print(f"✓ Done — JSON extraction complete")

In [ ]:
from google.colab import files
def evidence_tag(ev):
    if not ev or ev.strip().lower() == "not observed":
        return '<span style="color:#aaa;font-style:italic">not observed</span>'
    return f'<span style="color:#6b6a63;font-style:italic">"{ev}"</span>'

strengths_html = "".join(
    f"""<div class="finding strength">
         <div class="finding-point">&#10003; {s['point']}</div>
         <div class="finding-evidence">{evidence_tag(s.get('evidence',''))}</div>
       </div>"""
    for s in scaffolding.get("strengths", [])
)

growth_html = "".join(
    f"""<div class="finding growth">
         <div class="finding-point">&#9651; {g['point']}</div>
         <div class="finding-evidence">{evidence_tag(g.get('evidence',''))}</div>
       </div>"""
    for g in scaffolding.get("areas_for_growth", [])
)

scaffolding_section = f"""
<style>
  .summary-text {{ font-size: 14px; line-height: 1.7; color: #3d3d3a; margin-bottom: 18px; }}
  .finding {{ padding: 10px 14px; border-radius: 8px; margin-bottom: 8px; font-size: 13px; }}
  .strength {{ background: #f0faf5; border-left: 3px solid #1D9E75; }}
  .growth   {{ background: #fdf6ee; border-left: 3px solid #BA7517; }}
  .finding-point    {{ font-weight: 500; margin-bottom: 4px; }}
  .finding-evidence {{ font-size: 12px; }}
  .standout {{
    background: #eef4fb; border-radius: 8px; padding: 12px 16px;
    font-size: 13px; color: #185FA5; margin-top: 16px;
    border-left: 3px solid #185FA5;
  }}
  .two-col {{ display: flex; gap: 16px; }}
  .two-col > div {{ flex: 1; }}
</style>

<div class="section">
  <h2>&#9315; Scaffolding quality &nbsp;<span style="font-weight:400;color:#6b6a63">&#8212; Qwen2.5-7B analysis</span></h2>
  <p class="summary-text">{scaffolding.get('summary', '')}</p>
  <div class="two-col">
    <div>
      <strong style="font-size:12px;color:#6b6a63;text-transform:uppercase;letter-spacing:.04em">Strengths</strong>
      <br><br>{strengths_html}
    </div>
    <div>
      <strong style="font-size:12px;color:#6b6a63;text-transform:uppercase;letter-spacing:.04em">Areas for growth</strong>
      <br><br>{growth_html}
    </div>
  </div>
  <div class="standout">&#11088; Standout moment &#8212; {scaffolding.get('standout_moment', 'n/a')}</div>
</div>"""

# Call build_html_report to get the initial HTML string
initial_html_report = build_html_report(
    video_path, duration_s,
    filler_r, instruction_r, vocab_r,
    transcript,
)

updated_report = initial_html_report.replace("</body>", scaffolding_section + "\n</body>")

display(IHTML(updated_report))

final_report_path = "teaching_analysis_full_report.html"
with open(final_report_path, "w", encoding="utf-8") as f:
    f.write(updated_report)

print(f"✓ Full report saved → {final_report_path}")
files.download(final_report_path)
